In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [2]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

In [3]:
# 4-bit configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

c:\Users\ASUS TUF\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS TUF\.cache\huggingface\hub\models--Qwen--Qwen2.5-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [5]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

c:\Users\ASUS TUF\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS TUF\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [6]:
print("Model device:", model.device)

Model device: cuda:0


In [7]:
print("Memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2),
      "GB")

Memory allocated: 1.92 GB


In [8]:
print("Memory reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2),
      "GB")

Memory reserved: 1.98 GB


In [9]:
messages = [
    {
        "role": "system",
        "content": (
            "You analyze code-mixed Marathi-English social media text "
            "(Roman or Devanagari script). "
            "Reply with only a JSON object in exactly this format: "
            "{\"language\": \"mr-en\", "
            "\"sentiment\": \"positive|negative|neutral\", "
            "\"hate\": true|false, "
            "\"profanity\": true|false}. "
            "Set \"hate\" to true for hate speech and "
            "\"profanity\" to true for profane language, otherwise false."
        )
    },
    {
        "role": "user",
        "content": "Analyze this text.\nText: He bhalte avghad aste!"
    }
]

In [10]:
# Convert messages into Qwen's chat format
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

In [11]:
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

In [12]:
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

In [13]:
# Only decode the newly generated tokens
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

In [14]:
response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

In [15]:
print("Qwen response:")
print(response)

Qwen response:
{"language": "mr-en", "sentiment": "positive", "hate": true, "profanity": false}
